<a href="https://colab.research.google.com/github/GhostWolf09/Home_Assistant_MicroWakeWord_FR/blob/main/MicroWakeWord_mot_d'%C3%A9veil_perso_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### ⚙️ PARAMÈTRES ET NETTOYAGE
Définissez votre mot d'éveil ci-dessous et exécutez cette cellule pour initialiser les variables et effacer les anciennes données.

In [1]:
import os
import shutil
import glob

# @title ⚙️ CONFIGURATION DE VOTRE MOT D'ÉVEIL
WAKE_WORD = "Hey Liva" # @param {type:"string"}
PROJECT_NAME = "hey_liva_v5" # @param {type:"string"}
# Entrez les mots pièges séparés par des virgules pour éviter les faux positifs
MOTS_PIEGES = "Hey Lisa, Hey Linda, Hey Lida, Hey Rita, Hey Nina, Hey Tina, Hey Mina, Hey Silva, Hey Lola, Hey Lila, Hey Lova, Hey Léva, Hey Louva, Hey Livre, Hey Liam, Hey Lucas, Hey Leon, Hey Léna, Hey Lena, éléna, Hey Louis, elle y va, y va, elle va, il va, il y va, il y a, elle a, il a, et, et quoi, et la, hey la, lia, mia, ria, rita, chica, ia," # @param {type:"string"}

# Conversion de l'entrée texte en liste Python
TRAP_WORDS = [word.strip() for word in MOTS_PIEGES.split(',') if word.strip()]

print(f"🧹 Nettoyage de l'espace de travail pour le projet '{PROJECT_NAME}' en cours...")

# 1. Suppression des dossiers principaux et des modèles générés
dirs_to_remove = ['training_data', 'microWakeWord']
dirs_to_remove.extend(glob.glob(f'{PROJECT_NAME}_model_*'))

for d in dirs_to_remove:
    if os.path.exists(d):
        shutil.rmtree(d)
        print(f"🗑️ Dossier supprimé : {d}")

# 2. Suppression des fichiers temporaires
files_to_remove = (
    glob.glob('*.onnx') +
    glob.glob('*.onnx.json') +
    glob.glob('*.wav') +
    glob.glob('*.yaml') +
    glob.glob('*.py') +
    glob.glob('*.tar.gz') +
    glob.glob('*.zip') +
    glob.glob('*.tflite') +
    glob.glob('*.json') +
    ['input.txt']
)

for f in files_to_remove:
    if os.path.exists(f):
        os.remove(f)
        print(f"🗑️ Fichier supprimé : {f}")

print("\n✅ Configuration chargée et nettoyage terminé ! Vous pouvez passer à la suite.")

🧹 Nettoyage de l'espace de travail pour le projet 'hey_liva_v5' en cours...

✅ Configuration chargée et nettoyage terminé ! Vous pouvez passer à la suite.


### 1. Installation des dépendances
Nous installons `microWakeWord` et les outils nécessaires pour la synthèse vocale.

In [2]:
import os
# Installation depuis le GitHub spécifique en mode silencieux (-q)
!pip install -q git+https://github.com/OHF-Voice/micro-wake-word.git
!pip install -q huggingface_hub

# Création de la structure de dossiers
os.makedirs('training_data/wakeword', exist_ok=True)
os.makedirs('training_data/negative', exist_ok=True)
os.makedirs('training_data/background', exist_ok=True)
print("✅ Environnement prêt, nettoyé et optimisé !")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.5/84.5 kB 6.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.8/21.8 MB 105.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.1/86.1 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.6/119.6 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.2/83.2 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.1/82.1 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.2/52.2 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 

### 2. Importation de VOS enregistrements (Voix réelles)
Au lieu d'utiliser des voix générées par ordinateur, nous allons utiliser vos propres enregistrements pour un maximum de précision !

**💡 Combien de fichiers et comment les enregistrer ?**
*   **Quantité conseillée :** Entre 30 et 100 enregistrements de votre mot d'éveil. Plus vous en avez, plus le modèle sera robuste. (Format `.wav`)
*   **Variété des voix :** Faites participer vos proches ! Hommes, femmes, enfants.
*   **Intonations et émotions :** Parlez normalement, mais essayez aussi de le dire en étant fatigué, joyeux, en criant légèrement, ou en chuchotant.
*   **Débit de parole :** Dites le mot rapidement, puis lentement, en détachant bien les syllabes, ou au contraire en les liant.
*   **Environnement :** Variez la distance par rapport au micro (très près, à 1 mètre, à 3 mètres). Changez de pièce si possible (salon, cuisine, salle de bain pour l'écho).

**🎤 À propos de la qualité audio :**
L'idéal est de fournir des fichiers clairs (ex: 16kHz, mono). Cependant, **ce n'est pas grave si la qualité ou le format ne sont pas parfaits !** Le processus inclut un module de conversion et d'augmentation des données qui se chargera d'adapter et de nettoyer automatiquement vos fichiers pour l'entraînement.

Exécutez la cellule ci-dessous pour importer vos fichiers.

In [3]:
import os
from google.colab import files

output_dir = 'training_data/wakeword_diverse'
os.makedirs(output_dir, exist_ok=True)

print(f"\n📤 Cliquez sur 'Choisir des fichiers' ci-dessous pour uploader vos enregistrements .wav de '{WAKE_WORD}'")
print("💡 Astuce : Vous pouvez sélectionner tous vos fichiers d'un coup !")

uploaded = files.upload()

for filename, data in uploaded.items():
    if filename.endswith(".wav"):
        filepath = os.path.join(output_dir, filename)
        with open(filepath, 'wb') as f:
            f.write(data)

valides = [f for f in os.listdir(output_dir) if f.endswith('.wav')]
print(f"\n✅ Super ! {len(valides)} vrais enregistrements sont maintenant dans le dossier.")


📤 Cliquez sur 'Choisir des fichiers' ci-dessous pour uploader vos enregistrements .wav de 'Hey Liva'
💡 Astuce : Vous pouvez sélectionner tous vos fichiers d'un coup !


Saving hey_liva_01.wav to hey_liva_01.wav
Saving hey_liva_02.wav to hey_liva_02.wav
Saving hey_liva_03.wav to hey_liva_03.wav
Saving hey_liva_04.wav to hey_liva_04.wav
Saving hey_liva_05.wav to hey_liva_05.wav
Saving hey_liva_06.wav to hey_liva_06.wav
Saving hey_liva_07.wav to hey_liva_07.wav
Saving hey_liva_08.wav to hey_liva_08.wav
Saving hey_liva_09.wav to hey_liva_09.wav
Saving hey_liva_10.wav to hey_liva_10.wav
Saving hey_liva_11.wav to hey_liva_11.wav
Saving hey_liva_12.wav to hey_liva_12.wav
Saving hey_liva_13.wav to hey_liva_13.wav
Saving hey_liva_14.wav to hey_liva_14.wav
Saving hey_liva_15.wav to hey_liva_15.wav
Saving hey_liva_16.wav to hey_liva_16.wav
Saving hey_liva_17.wav to hey_liva_17.wav
Saving hey_liva_18.wav to hey_liva_18.wav
Saving hey_liva_19.wav to hey_liva_19.wav
Saving hey_liva_20.wav to hey_liva_20.wav
Saving hey_liva_21.wav to hey_liva_21.wav
Saving hey_liva_22.wav to hey_liva_22.wav
Saving hey_liva_23.wav to hey_liva_23.wav
Saving hey_liva_24.wav to hey_liva

In [4]:
import os
import glob

wakeword_dir = 'training_data/wakeword_diverse'
wav_files = glob.glob(os.path.join(wakeword_dir, '*.wav'))

print(f"📂 Nombre total de fichiers dans '{wakeword_dir}' : {len(wav_files)}")

empty_files = [f for f in wav_files if os.path.getsize(f) <= 44]
print(f"⚠️ Nombre de fichiers vides/corrompus : {len(empty_files)}")

if len(wav_files) > 0 and len(empty_files) == 0:
    print("\n✅ Parfait ! Vos fichiers audio de base sont valides. Vous pouvez passer à l'étape suivante (Génération des mots pièges) !")
else:
    print("\n⚠️ Erreur : Aucun fichier de base trouvé ou il y a des fichiers corrompus.")

📂 Nombre total de fichiers dans 'training_data/wakeword_diverse' : 53
⚠️ Nombre de fichiers vides/corrompus : 0

✅ Parfait ! Vos fichiers audio de base sont valides. Vous pouvez passer à l'étape suivante (Génération des mots pièges) !


In [5]:
!pip install -q piper-tts
import os
import subprocess
from tqdm import tqdm

# Téléchargement du modèle vocal Piper juste pour les mots pièges
model = "siwis"
url_base = "https://huggingface.co/rhasspy/piper-voices/resolve/main/fr/fr_FR/siwis/medium/fr_FR-siwis-medium"
if not os.path.exists(f"{model}.onnx"):
    print("⬇️ Téléchargement de la voix de synthèse pour les mots pièges...")
    os.system(f"wget -q {url_base}.onnx -O {model}.onnx")
    os.system(f"wget -q {url_base}.onnx.json -O {model}.onnx.json")

hard_negatives_dir = 'training_data/hard_negatives'
os.makedirs(hard_negatives_dir, exist_ok=True)

print(f"\n🎙️ Génération des {len(TRAP_WORDS)} mots pièges de base avec Piper...")

for i, word in enumerate(tqdm(TRAP_WORDS, desc="Génération des mots pièges")):
    filename = os.path.join(hard_negatives_dir, f"base_neg_{i}.wav")
    # Génération du mot piège
    cmd = f'echo "{word}" | piper -m {model}.onnx -f {filename} 2>/dev/null'
    subprocess.run(cmd, shell=True)

print(f"\n✅ Mots pièges de base générés dans {hard_negatives_dir} !")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.1/34.1 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 110.9 MB/s eta 0:00:00
⬇️ Téléchargement de la voix de synthèse pour les mots pièges...

🎙️ Génération des 39 mots pièges de base avec Piper...


Génération des mots pièges: 100%|██████████| 39/39 [01:49<00:00,  2.81s/it]


✅ Mots pièges de base générés dans training_data/hard_negatives !


### 3. Expansion de timbre avec PyWorld

Cette section installe les dépendances requises, génère le script d'augmentation de données, et l'applique sur vos enregistrements de base (`training_data/wakeword_diverse`). Les nouveaux fichiers seront sauvegardés directement dans le **même dossier** afin d'être automatiquement pris en compte lors de la conversion (Étape 4).

#### 3.1 Préparation de l'environnement audio
Nous installons les bibliothèques spécialisées nécessaires, comme `pyworld` (pour extraire et modifier les caractéristiques de la voix) et `librosa` (pour l'analyse et le traitement du son).

In [6]:
# Installation des dépendances
!pip install soundfile pyworld numpy librosa --break-system-packages


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.0/261.0 kB 7.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pyworld: filename=pyworld-0.3.5-cp313-cp313-linux_x86_64.whl size=1019258 sha256=870e5ead0a013fd36e775ac888f1b4db76019764c708948b04361f237863d917
  Stored in directory: /root/.cache/pip/wheels/8b/c7/b2/0fb039381aa5f5b31c58216a65288315fda280becb7ae8f991
Successfully built pyworld


#### 3.2 Création du script de transformation vocale
Ce bloc écrit le code du fichier `expansion_timbre.py`. C'est le « moteur » qui contient les règles mathématiques pour altérer vos enregistrements. Il modifie la fréquence fondamentale (pitch) pour simuler différents âges et genres, et applique des variations de vitesse (time-stretching).

In [7]:
import os

# Création du script expansion_timbre.py
script_content = """
import os
import sys
import glob
import argparse
import numpy as np
import soundfile as sf
import pyworld as pw
import librosa

def process_audio(input_path, output_dir):
    # Charger l'audio et forcer 16kHz mono
    y, sr = librosa.load(input_path, sr=16000, mono=True)
    y = y.astype(np.float64)

    # Extraire les caractéristiques vocales avec PyWorld
    _f0, t = pw.harvest(y, sr)
    sp = pw.cheaptrick(y, _f0, t, sr)
    ap = pw.d4c(y, _f0, t, sr)

    base_name = os.path.splitext(os.path.basename(input_path))[0]

    # Définition des variations de timbre (Pitch multiplier)
    # > 1.0 = voix plus aiguë (enfant/femme), < 1.0 = voix plus grave (homme/âgé)
    timbre_variations = {
        'enfant': 1.6,
        'femme_aigue': 1.3,
        'femme': 1.15,
        'homme_clair': 0.9,
        'homme_grave': 0.75,
        'age': 0.6
    }

    # Définition des variations de vitesse (Speed multiplier)
    # > 1.0 = plus rapide, < 1.0 = plus lent
    speed_variations = {
        'lent': 0.85,
        'normal': 1.0,
        'rapide': 1.15
    }

    # Générer les variantes
    count = 0
    for t_name, f0_mult in timbre_variations.items():
        for s_name, speed_mult in speed_variations.items():
            # Modifier le pitch
            f0_mod = _f0 * f0_mult

            # Synthétiser la nouvelle voix
            y_mod = pw.synthesize(f0_mod, sp, ap, sr)

            # Modifier la vitesse si nécessaire
            if speed_mult != 1.0:
                y_mod = librosa.effects.time_stretch(y_mod, rate=speed_mult)

            # Normalisation audio
            max_val = np.max(np.abs(y_mod))
            if max_val > 0:
                y_mod = y_mod / max_val * 0.95

            # Sauvegarder en 16-bit PCM mono 16kHz
            out_path = os.path.join(output_dir, "{}_{}_{}.wav".format(base_name, t_name, s_name))
            sf.write(out_path, y_mod, sr, subtype='PCM_16')
            count += 1

    return count

if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('input_dir', help='Dossier source')
    parser.add_argument('output_dir', help='Dossier de destination')
    args = parser.parse_args()

    os.makedirs(args.output_dir, exist_ok=True)

    wav_files = glob.glob(os.path.join(args.input_dir, '*.wav'))
    print("Traitement de {} fichiers...".format(len(wav_files)))

    total_generated = 0
    for f in wav_files:
        try:
            total_generated += process_audio(f, args.output_dir)
        except Exception as e:
            print("Erreur sur {}: {}".format(f, e))

    print("Succès ! {} nouveaux échantillons générés dans {}".format(total_generated, args.output_dir))
"""

with open('expansion_timbre.py', 'w') as f:
    f.write(script_content)

print("✅ Script 'expansion_timbre.py' créé avec succès.")

✅ Script 'expansion_timbre.py' créé avec succès.


#### 3.3 Lancement de l'augmentation des données
C'est l'étape finale où la magie opère ! Le script s'exécute sur vos fichiers.
*💡 Note importante : Les nouvelles voix générées sont enregistrées directement au même endroit que vos fichiers originaux. Ainsi, l'Étape 4 les prendra automatiquement en compte !*

In [8]:
# 1. Expansion du mot d'éveil principal
wakeword_dir = "training_data/wakeword_diverse"
print("🚀 Lancement de l'expansion de timbre pour le MOT D'ÉVEIL...")
!python3 expansion_timbre.py {wakeword_dir} {wakeword_dir}

# 2. Expansion des mots pièges (hard_negatives)
trap_words_dir = "training_data/hard_negatives"
print("\n🚀 Lancement de l'expansion de timbre pour les MOTS PIÈGES...")
!python3 expansion_timbre.py {trap_words_dir} {trap_words_dir}

# Afficher un résumé
import glob
total_wakewords = len(glob.glob(f"{wakeword_dir}/*.wav"))
total_trap_words = len(glob.glob(f"{trap_words_dir}/*.wav"))
print(f"\n📊 RÉSUMÉ FINAL :")
print(f" - Mots d'éveil : {total_wakewords} fichiers au total.")
print(f" - Mots pièges  : {total_trap_words} fichiers au total.")
print("\n➡️ Vous pouvez maintenant lancer la suite !")

🚀 Lancement de l'expansion de timbre pour le MOT D'ÉVEIL...
/usr/local/lib/python3.13/dist-packages/pyworld/__init__.py:13: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Traitement de 53 fichiers...
Succès ! 954 nouveaux échantillons générés dans training_data/wakeword_diverse

🚀 Lancement de l'expansion de timbre pour les MOTS PIÈGES...
/usr/local/lib/python3.13/dist-packages/pyworld/__init__.py:13: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Traitement de 39 fichiers...
Succès ! 702 nouveaux échantillons générés dans training_data/hard_negatives

📊 RÉS

### 4. Téléchargement des données négatives depuis Hugging Face
Nous téléchargeons les backgrounds et les mots négatifs depuis le dataset `kahrendt/microwakeword`.

In [9]:
import os
import glob
import zipfile
from huggingface_hub import snapshot_download

print("⬇️ Téléchargement des archives (.zip) depuis Hugging Face...")

snapshot_download(
    repo_id="kahrendt/microwakeword",
    repo_type="dataset",
    local_dir="training_data",
    allow_patterns=["*.zip"]
)

print("\n📦 Extraction des fichiers .zip (cela peut prendre quelques minutes)...")
zip_files = glob.glob("training_data/*.zip")

if not zip_files:
    print("⚠️ Aucun fichier .zip trouvé dans le dossier ! Vérifiez la connexion ou le dépôt.")
else:
    for z in zip_files:
        print(f" -> Extraction de {os.path.basename(z)}...")
        with zipfile.ZipFile(z, 'r') as zip_ref:
            zip_ref.extractall("training_data")
        # Nettoyage du .zip pour libérer de l'espace disque sur Colab
        os.remove(z)

print("\n✅ Terminé ! Les dossiers mmap sont maintenant extraits et prêts.")


⬇️ Téléchargement des archives (.zip) depuis Hugging Face...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]


📦 Extraction des fichiers .zip (cela peut prendre quelques minutes)...
 -> Extraction de no_speech_background.zip...
 -> Extraction de dinner_party_background.zip...
 -> Extraction de dinner_party_eval.zip...
 -> Extraction de no_speech.zip...
 -> Extraction de speech_background.zip...
 -> Extraction de dinner_party.zip...
 -> Extraction de speech.zip...

✅ Terminé ! Les dossiers mmap sont maintenant extraits et prêts.


### 5. Conversion des données en Spectrogrammes (.mmap)
Cette étape transforme les milliers de fichiers audio (.wav) en empreintes visuelles (spectrogrammes) au format `.mmap`. Cela permet au réseau de neurones d'ingérer les données à très haute vitesse pendant l'entraînement sans saturer la mémoire vive de Colab.

In [10]:
import sys
import os
import glob
import gc
import shutil

# @title ⚙️ CONFIGURATION DE L'AUGMENTATION
# @markdown Choisissez le nombre total d'échantillons à générer artificiellement.
# @markdown *Conseil : Pour Colab Gratuit, gardez ces valeurs autour de 10000 à 15000 pour éviter de saturer la RAM.*
TARGET_WAKEWORD_COUNT = 100000 # @param {type:"integer"}
TARGET_NEGATIVE_COUNT = 60000 # @param {type:"integer"}

# --- CLONAGE DU DEPOT SI MANQUANT ---
if not os.path.exists('/content/microWakeWord'):
    print("📥 Téléchargement du dépôt microWakeWord manquant...")
    os.system("git clone https://github.com/OHF-Voice/micro-wake-word.git /content/microWakeWord")
# ------------------------------------

# On force Python à oublier la version buggée en mémoire
for key in list(sys.modules.keys()):
    if key.startswith('microwakeword'):
        del sys.modules[key]

sys.path.insert(0, '/content/microWakeWord')

import soundfile as sf
from mmap_ninja.ragged import RaggedMmap
from microwakeword.audio.clips import Clips
from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.spectrograms import SpectrogramGeneration

# 1. On récupère les bruits de fond téléchargés à l'étape 3 pour les mélanger aux voix
bg_paths = glob.glob('training_data/*_background/**/*.wav', recursive=True) + glob.glob('training_data/dinner_party/**/*.wav', recursive=True)

# 2. Configuration de l'augmentation de données dynamique native
print("🎛️ Configuration de l'augmentation de données dynamique (microWakeWord)...")
augmenter = Augmentation(
    augmentation_duration_s=3.2,
    augmentation_probabilities={
        "PitchShift": 0.5,
        "TimeStretch": 0.5,
        "Gain": 0.5,
        "AddGaussianNoise": 0.3,
        "AddBackgroundNoise": 0.5 if len(bg_paths) > 0 else 0.0
    },
    impulse_paths=[],
    background_paths=bg_paths
)

dirs_to_convert = [
    ('training_data/wakeword_diverse', 'training_data/wakeword_mmap'),
    ('training_data/kahrendt_dataset', 'training_data/background_mmap'),
    ('training_data/hard_negatives', 'training_data/hard_negatives_mmap')
]

print("⚙️ DÉBUT DE LA CONVERSION EN .MMAP...\n")

for in_dir, out_dir in dirs_to_convert:
    if not os.path.exists(in_dir) or len(os.listdir(in_dir)) == 0:
        if 'kahrendt_dataset' in in_dir:
            print(f"✅ Info : Le dossier {in_dir} est géré automatiquement (c'est normal qu'il soit absent).")
        else:
            print(f"⚠️ Dossier ignoré car vide ou inexistant : {in_dir}")
        continue

    # NOUVEAU : Suppression de l'ancien dossier pour forcer la mise à jour avec les nouvelles voix
    if os.path.exists(out_dir):
        print(f"♻️ Nettoyage de l'ancien dossier {out_dir} pour forcer la mise à jour...")
        shutil.rmtree(out_dir)

    os.makedirs(out_dir, exist_ok=True)
    mmap_path = os.path.join(out_dir, 'features_mmap')

    print(f"🧹 Vérification et nettoyage des fichiers corrompus dans {in_dir}...")
    wav_files = glob.glob(os.path.join(in_dir, '**/*.wav'), recursive=True)
    for f in wav_files:
        try:
            with sf.SoundFile(f) as _:
                pass
        except Exception:
            os.remove(f)

    # --- DÉTERMINATION DU FACTEUR DE RÉPÉTITION ---
    # Au lieu de générer N wavs sur le disque, on dit à microWakeWord de repasser sur les mêmes fichiers
    # plusieurs fois (repeat). À chaque passage, il applique l'augmentation aléatoire de façon différente !
    file_count = len(glob.glob(os.path.join(in_dir, '**/*.wav'), recursive=True))
    repeat_factor = 1

    if 'wakeword_diverse' in in_dir and file_count > 0:
        repeat_factor = max(1, TARGET_WAKEWORD_COUNT // file_count)
        print(f"🔄 Objectif Wakeword : {file_count} fichiers physiques. Répétition x{repeat_factor} (total : ~{file_count * repeat_factor} générations dynamiques).")
    elif 'hard_negatives' in in_dir and file_count > 0:
        repeat_factor = max(1, TARGET_NEGATIVE_COUNT // file_count)
        print(f"🔄 Objectif Négatifs : {file_count} fichiers physiques. Répétition x{repeat_factor} (total : ~{file_count * repeat_factor} générations dynamiques).")

    print(f"↻ Conversion de {in_dir} vers {mmap_path}...")

    clips = Clips(input_directory=in_dir,
                  file_pattern='**/*.wav',
                  max_clip_duration_s=None,
                  remove_silence=False,
                  split_count=0)

    spectrograms = SpectrogramGeneration(clips=clips,
                                         augmenter=augmenter,
                                         slide_frames=1,
                                         step_ms=10)

    # Générateur personnalisé pour afficher la progression
    def progress_generator(gen, folder_name):
        for i, item in enumerate(gen):
            if i > 0 and i % 500 == 0:
                print(f"  ⏳ [{folder_name}] {i} spectrogrammes générés...")
            yield item

    RaggedMmap.from_generator(
        out_dir=mmap_path,
        sample_generator=progress_generator(spectrograms.spectrogram_generator(split=None, repeat=repeat_factor), os.path.basename(in_dir)),
        batch_size=100,
        verbose=False # Mis sur False pour éviter l'erreur de log UTF-8
    )
    gc.collect() # Libération de la RAM après chaque dossier

print("\n✅ TOUTES LES CONVERSIONS SONT TERMINÉES !")

📥 Téléchargement du dépôt microWakeWord manquant...
🎛️ Configuration de l'augmentation de données dynamique (microWakeWord)...
⚙️ DÉBUT DE LA CONVERSION EN .MMAP...

🧹 Vérification et nettoyage des fichiers corrompus dans training_data/wakeword_diverse...
🔄 Objectif Wakeword : 1007 fichiers physiques. Répétition x99 (total : ~99693 générations dynamiques).
↻ Conversion de training_data/wakeword_diverse vers training_data/wakeword_mmap/features_mmap...
  ⏳ [wakeword_diverse] 500 spectrogrammes générés...
  ⏳ [wakeword_diverse] 1000 spectrogrammes générés...
  ⏳ [wakeword_diverse] 1500 spectrogrammes générés...
  ⏳ [wakeword_diverse] 2000 spectrogrammes générés...
  ⏳ [wakeword_diverse] 2500 spectrogrammes générés...
  ⏳ [wakeword_diverse] 3000 spectrogrammes générés...
  ⏳ [wakeword_diverse] 3500 spectrogrammes générés...
  ⏳ [wakeword_diverse] 4000 spectrogrammes générés...
  ⏳ [wakeword_diverse] 4500 spectrogrammes générés...
  ⏳ [wakeword_diverse] 5000 spectrogrammes générés...
  ⏳ [

In [11]:
import os
import yaml
import time
import shutil
import glob

# @title ⚙️ CONFIGURATION DE L'ENTRAÎNEMENT
# @markdown Paliers optimisés pour une **excellente qualité**.
STEP_1 = 60000 # @param {type:"integer"}
STEP_2 = 80000 # @param {type:"integer"}
STEP_3 = 100000 # @param {type:"integer"}
# @markdown ---
# @markdown ⚠️ **ATTENTION MÉMOIRE (RAM)** : L'entraînement consomme énormément de RAM (jusqu'à 45 Go avec beaucoup de données).
# @markdown * **Colab Gratuit (~13 Go RAM)** : Mettez le `BATCH_SIZE` à 32 ou 64. (Vous devez aussi avoir réduit les cibles générées à l'étape 5 autour de 8000).
# @markdown * **Colab Pro** : Vous pouvez laisser à 128 ou 256.
BATCH_SIZE = 256 # @param {type:"integer"}

TOTAL_STEPS = STEP_1 + STEP_2 + STEP_3
print(f"\n⏱️ Info : L'entraînement s'arrêtera après un total de {TOTAL_STEPS} étapes (paliers : {STEP_1}, {STEP_2}, {STEP_3}).")
print(f"📦 Taille des lots (Batch Size) : {BATCH_SIZE}")

# --- RESTAURATION DU MOTEUR KERAS 3 ---
if "TF_USE_LEGACY_KERAS" in os.environ:
    del os.environ["TF_USE_LEGACY_KERAS"]
os.environ["KERAS_BACKEND"] = "tensorflow"

# --- INSTALLATION DES DEPENDANCES MANQUANTES ---
!pip install -q mmap_ninja audio_metadata webrtcvad pymicro-features audiomentations ai-edge-litert
# -----------------------------------------------

# --- CLONAGE OU RESTAURATION DU DEPOT ---
if not os.path.exists('/content/microWakeWord'):
    print("Téléchargement du dépôt microWakeWord...")
    !git clone https://github.com/OHF-Voice/micro-wake-word.git /content/microWakeWord
else:
    print("Restauration du code source original de microWakeWord (Keras 3)...")
    !cd /content/microWakeWord && git restore .
# ------------------------------------

print("\n⚙️ Vérification de l'arborescence des dossiers (Training / Testing / Validation)...")
for folder in ['wakeword_mmap', 'hard_negatives_mmap']:
    base_dir = f'training_data/{folder}'
    old_path = f'{base_dir}/features_mmap'
    train_dir = f'{base_dir}/training/features_mmap'
    test_dir = f'{base_dir}/testing/features_mmap'
    val_dir = f'{base_dir}/validation/features_mmap'

    if os.path.exists(old_path) and not os.path.exists(train_dir):
        os.makedirs(f'{base_dir}/training', exist_ok=True)
        shutil.move(old_path, train_dir)
        print(f" -> Dossier d'entraînement créé : {train_dir}")

    if os.path.exists(train_dir):
        if not os.path.exists(test_dir):
            shutil.copytree(train_dir, test_dir)
        if not os.path.exists(val_dir):
            shutil.copytree(train_dir, val_dir)

print("\n🔍 Recherche des datasets valides...")
features_list = []

for item in os.listdir('training_data'):
    base_path = os.path.join('training_data', item)
    if os.path.isdir(base_path) and os.path.isdir(os.path.join(base_path, 'training')):
        if 'wakeword' in item.lower():
            features_list.append({
                "features_dir": base_path,
                "penalty_weight": 1.0,
                "sampling_weight": 2.0,
                "truncation_strategy": "truncate_start",
                "truth": True,
                "type": "mmap"
            })
        elif item in ['speech', 'no_speech', 'hard_negatives_mmap']:
            features_list.append({
                "features_dir": base_path,
                "penalty_weight": 1.0,
                "sampling_weight": 10.0,
                "truncation_strategy": "random",
                "truth": False,
                "type": "mmap"
            })
        else:
            features_list.append({
                "features_dir": base_path,
                "penalty_weight": 1.0,
                "sampling_weight": 5.0,
                "truncation_strategy": "random",
                "truth": False,
                "type": "mmap"
            })

# 3. Generer un dossier de sortie unique avec le nom du projet
output_dir = f"{PROJECT_NAME}_model_{int(time.time())}"
os.makedirs(output_dir, exist_ok=True)

print("\n📝 Etape 1 : Generation du fichier de configuration YAML...")
config = {
    "clip_duration_ms": 1490,
    "window_step_ms": 10,
    "train_dir": output_dir,
    "model_architecture": "inception",
    "model_size_info": [5, 256, 256],
    "batch_size": BATCH_SIZE,
    "eval_step_interval": 1000,
    "minimization_metric": "loss",
    "target_minimization": 0.0,
    "maximization_metric": "accuracy",
    "target_maximization": 1.0,
    "training_steps": [STEP_1, STEP_2, STEP_3],
    "learning_rates": [0.001, 0.0005, 0.0002],
    "features": features_list
}

with open("training_parameters.yaml", "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print("Fichier YAML cree avec succes !")
print(f"Les modeles entraines seront sauvegardes dans : {output_dir}")

print("\n🚀 Etape 2 : Lancement de l'entrainement...")

# On utilise tee pour afficher dans la console tout en sauvegardant dans un fichier log
!PYTHONPATH=/content/microWakeWord python -m microwakeword.model_train_eval \
  --training_config training_parameters.yaml \
  --train 1 \
  --restore_checkpoint 1 \
  inception 2>&1 | tee training_eval.log

import re

# Vérification stricte de la génération du TFLite
tflite_files = glob.glob(f"{output_dir}/**/*.tflite", recursive=True)

if not tflite_files:
    print("\n" + "❌"*30)
    print("❌ ERREUR CRITIQUE : L'entraînement n'est pas allé jusqu'au bout !")
    print("❌ Google Colab a probablement manqué de RAM et a coupé le processus.")
    print("❌ Veuillez réessayer avec des valeurs de STEP ou de BATCH_SIZE plus basses.")
    print("❌"*30)
else:
    print("\n" + "="*60)
    print("🎉 ENTRAÎNEMENT TERMINÉ AVEC SUCCÈS ! FICHIER TFLITE GÉNÉRÉ ! 🎉")
    print("="*60)

    try:
        with open("training_eval.log", "r") as f:
            log_content = f.read()

        print("\n📊 RÉSULTATS THÉORIQUES DU MODÈLE :\n")

        cutoffs = re.findall(r"Cutoff ([0-9.]+): frr=([0-9.]+); faph=([0-9.]+)", log_content)
        if cutoffs:
            print("🎯 Statistiques estimées selon le seuil de confiance (Cutoff) :\n")
            print("  [ FRR = % de fois où l'appareil ne vous entendra pas (Faux Rejet) ]")
            print("  [ FAPH = Nombre de déclenchements fantômes par heure (Faux Positifs) ]\n")
            for cutoff, frr, faph in cutoffs:
                print(f"   ▶ Seuil {cutoff}  =>  FRR : {float(frr)*100:05.2f}%  |  FAPH : {faph} / heure")
        else:
            print("   (Statistiques détaillées non trouvées dans le log)")

        print("\n💡 INTERPRÉTATION :")
        print(" ➡️ Un seuil (Cutoff) ÉLEVÉ (ex: 0.80) réduit les faux déclenchements (FAPH), mais l'assistant pourrait parfois vous ignorer (FRR plus haut).")
        print(" ➡️ Un seuil (Cutoff) FAIBLE (ex: 0.40) garantit qu'il vous entendra toujours, mais il risque de s'activer tout seul avec des mots proches (mots pièges).")
        print("\n🛠️ CONSEIL FINAL :")
        print("Ne vous inquiétez pas si ces chiffres ne semblent pas parfaits ! Les simulateurs sont souvent plus sévères que la réalité.")
        print("Le vrai test se fait sur votre appareil. Vous pourrez toujours ajuster la variable 'probability_cutoff' dans votre fichier ESPHome pour trouver le juste milieu.")
        print("\n➡️ Passez à l'étape 6 pour télécharger vos fichiers !")

    except Exception as e:
        print(f"\n⚠️ Impossible d'analyser les logs pour les statistiques : {e}")
        print("Cependant, l'entraînement est bien terminé ! Vous pouvez passer à la cellule d'export.")


⏱️ Info : L'entraînement s'arrêtera après un total de 240000 étapes (paliers : 60000, 80000, 100000).
📦 Taille des lots (Batch Size) : 256
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
Restauration du code source original de microWakeWord (Keras 3)...

⚙️ Vérification de l'arborescence des dossiers (Training / Testing / Validation)...
 -> Dossier d'entraînement créé : training_data/wakeword_mmap/training/features_mmap
 -> Dossier d'entraînement créé : training_data/hard_negatives_mmap/training/features_mmap

🔍 Recherche des datasets valides...

📝 Etape 1 : Generation du fichier de configuration YAML...
Fichier YAML cree avec succes !
Les modeles entraines seront sauvegardes dans : hey_liva_v5_model_1791729301

🚀 Etape 2 : Lancement de l'entrainement...
I0000 00:00:1791729301.881918   20530 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-

### 6. Export et téléchargement des fichiers finaux
Cette cellule isole les fichiers générés par l'entraînement (`.tflite` et `.json`), les renomme correctement et lance leur téléchargement direct.

In [12]:
import os
import glob
import json
from google.colab import files

# --- RESTAURATION AUTOMATIQUE DES VARIABLES ---
# Si la session Colab a redémarré en mémoire, on récupère les infos via les dossiers
if 'PROJECT_NAME' not in locals() and 'PROJECT_NAME' not in globals():
    dossiers = sorted(glob.glob('*_model_*'))
    if dossiers:
        PROJECT_NAME = dossiers[-1].split('_model_')[0]
        print(f"🔄 Nom du projet restauré automatiquement : {PROJECT_NAME}")
    else:
        PROJECT_NAME = "hey_liva" # Valeur par défaut de secours

if 'WAKE_WORD' not in locals() and 'WAKE_WORD' not in globals():
    WAKE_WORD = PROJECT_NAME.replace('_', ' ').title()
    print(f"🔄 Mot d'éveil restauré automatiquement : {WAKE_WORD}")
# ----------------------------------------------

# Recherche automatique du dossier de modèle le plus récent avec le nom du projet
model_dirs = sorted(glob.glob(f'{PROJECT_NAME}_model_*'))
if not model_dirs:
    print("❌ Aucun dossier de modèle trouvé.")
else:
    output_dir = model_dirs[-1]
    print(f"📂 Dossier d'entraînement le plus récent trouvé : {output_dir}")

    tflite_filename = f'{PROJECT_NAME}.tflite'
    json_filename = f'{PROJECT_NAME}.json'

    # 1. Recherche et renommage du modèle TFLite
    tflite_files = glob.glob(os.path.join(output_dir, '**', '*.tflite'), recursive=True)
    if tflite_files:
        tflite_path = tflite_files[0]
        os.system(f'cp "{tflite_path}" /content/{tflite_filename}')
        print(f"✅ Fichier TFLite trouvé et préparé : {tflite_filename}")
    else:
        print("❌ Aucun fichier .tflite n'a été trouvé. L'entraînement a peut-être échoué.")

    # 2. Préparation du fichier JSON ESPHome
    esphome_manifest = {
        "type": "micro",
        "wake_word": WAKE_WORD,
        "author": "Utilisateur Colab",
        "website": "https://github.com/",
        "model": tflite_filename,
        "trained_languages": [
            "fr"
        ],
        "version": 2,
        "micro": {
            "probability_cutoff": 0.85,
            "feature_step_size": 10,
            "sliding_window_size": 5,
            "tensor_arena_size": 50000,
            "minimum_esphome_version": "2024.7.0"
        }
    }

    with open(f'/content/{json_filename}', 'w') as f:
        json.dump(esphome_manifest, f, indent=4)
    print(f"✅ Fichier {json_filename} généré avec succès.")

    # 3. Téléchargement direct
    print("\n⬇️ Lancement des téléchargements...")
    if os.path.exists(f'/content/{tflite_filename}'):
        files.download(f'/content/{tflite_filename}')
    if os.path.exists(f'/content/{json_filename}'):
        files.download(f'/content/{json_filename}')

📂 Dossier d'entraînement le plus récent trouvé : hey_liva_v5_model_1791729301
✅ Fichier TFLite trouvé et préparé : hey_liva_v5.tflite
✅ Fichier hey_liva_v5.json généré avec succès.

⬇️ Lancement des téléchargements...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 7. 🚀 Comment utiliser votre modèle sur ESP32 (ESPHome)

Félicitations, vous avez maintenant vos précieux fichiers `.tflite` et `.json` ! Voici comment les intégrer à votre assistant vocal ESP32 :

**1. Où placer les fichiers ?**
Transférez les deux fichiers téléchargés dans le répertoire de configuration de votre ESPHome. Selon votre installation (Home Assistant, Docker, etc.), cela se trouve généralement au même endroit que vos fichiers `.yaml` (parfois dans un sous-dossier spécifique selon vos habitudes).

**2. Configuration ESPHome (.yaml)**
Dans le fichier de configuration de votre ESP32, vous devez indiquer le chemin vers votre modèle. Assurez-vous que le bloc `micro_wake_word` est bien configuré pour utiliser ces nouveaux fichiers. Ajoutez ou modifiez ces lignes :

```yaml
micro_wake_word:
  models:
    - model: models/votre_modele.json # Remplacez par le chemin exact de votre fichier .json
```

**3. 🎛️ Ajustement de la sensibilité**
Il est très fréquent que le modèle réagisse un peu différemment dans le monde réel (à cause du micro de l'ESP32, de la réverbération de la pièce, etc.).
Si besoin, ouvrez le fichier **`.json`** que vous venez de télécharger avec un simple éditeur de texte (Bloc-notes, TextEdit, etc.) et cherchez cette ligne :

```json
"probability_cutoff":
```

*   **L'assistant a du mal à vous entendre (trop sourd) ?** Baissez la valeur (ex: `0.70`, `0.60` ou `0.50`). Il sera plus sensible pour la détection.
*   **L'assistant s'active tout seul avec des mots proches (faux positifs) ?** Augmentez la valeur (ex: `0.90` ou `0.95`). Il sera plus strict et exigera une prononciation parfaite.

*Note : À chaque modification de ce fichier `.json`, vous devrez relancer l'installation (OTA/USB) sur votre ESP32 pour que le nouveau réglage soit pris en compte.*